# CTF 抢旗 Bot

## 1. 初始化依赖


In [1]:
import json
import random

from javascript import require, On, Once, AsyncTask, once, off

mineflayer = require("mineflayer")

pathfinder = require("mineflayer-pathfinder")
Vec3 = require("vec3").Vec3

## 2. 连接服务器


In [2]:
HOST = "10.31.0.101"
TEAM_NUM = 1029
YOUR_NAME = 1
AGAINST_TEAM = "none"  # Example: 1, 2, ... or None for debug mode
PER_TEAM_PLAYER = 1
MAP_MODE = "fixed"  # "fixed" or "random"

team_message = {}

bot = mineflayer.createBot({"host": HOST, "port": 25565, "username": f"CTF-{TEAM_NUM}-{YOUR_NAME}", "hideErrors": False})
bot.loadPlugin(pathfinder.pathfinder)

once(bot, "spawn")

[]

### 加入游戏

In [ ]:
bot.chat(f"with {AGAINST_TEAM} {PER_TEAM_PLAYER} {MAP_MODE}")

game_start = False
all_messages = []

@On(bot, "messagestr")
def on_message(this, message, *rest):
    # Depending on mineflayer version, 
    # sometimes @this is the message and @message is the sender
    # sometimes @message is the message and @this is the sender
    global game_start
    global all_messages
    msg = f"{this} {message}"
    all_messages.append(msg)
    maybe_sender = str(this)
    maybe_message = str(message)

    #all_messages.append(f"{this} / {message} : {full_message}")
    if "Are you ready?" in maybe_sender or "Are you ready?" in maybe_message:
        bot.chat("I'm ready!")
    elif maybe_message.startswith("Game start: "):
        game_start = True
        team_message.update(json.loads(message[12:]))
    elif maybe_sender.startswith("Game start: "):
        game_start = True
        team_message.update(json.loads(this[12:]))
    elif "Game over!" in maybe_sender or "Game over!" in maybe_message:
        game_start = False

## 3. 加载寻路

In [5]:
movements = pathfinder.Movements(bot)
movements.canDig = False
movements.allow1by1towers = False
movements.allowParkour = False
movements.allowSprinting = True
bot.pathfinder.setMovements(movements)

## 4. 确定自己是哪一队


In [6]:
def decide_my_team(bot):
    if bot.entity.username in team_message["left"]:
        my_team = "L"
        home_banner = "red_banner"
        enemy_banner = "blue_banner"
        print("I am on team L")
    elif bot.entity.username in team_message["right"]:
        my_team = "R"
        home_banner = "blue_banner"
        enemy_banner = "red_banner"
        print("I am on team R")
    else:
        my_team = None
        home_banner = None
        enemy_banner = None
        print("I am not on a team")
    return (my_team, home_banner, enemy_banner)

my_team, home_banner, enemy_banner = decide_my_team(bot)

I am on team R


## 5. 寻找主要blocks：
- "gold_block": 可放置旗子的位置（只在自己的场地有效）
- "stone_pressure_plate"，z坐标为24: 可解救队友的位置（在双方场地均有效）


In [7]:
def find_blocks_by_name(name, max_distance=64, count=16):
    try:
        block_id = int(bot.registry.blocksByName[name].id)
    except Exception:
        return []
    found = bot.findBlocks({"matching": block_id, "maxDistance": max_distance, "count": count})
    return sorted({(int(p.x), int(p.y), int(p.z)) for p in found})

find_blocks_by_name("gold_block")

[(-22, 0, 6),
 (-22, 0, 10),
 (-22, 0, 14),
 (-22, 0, 18),
 (-22, 0, 22),
 (-22, 0, 26),
 (-22, 0, 30),
 (-22, 0, 34),
 (22, 0, 6),
 (22, 0, 10),
 (22, 0, 14),
 (22, 0, 18),
 (22, 0, 22),
 (22, 0, 26),
 (22, 0, 30),
 (22, 0, 34)]

## 6. 观察旗子、背包和交旗点


In [8]:
def carrying_enemy_flag():
    if enemy_banner is None:
        return False
    for item in bot.inventory.slots:
        # print(item)
        if item and item.name == enemy_banner:
            print(item.name)
            return True
    return False

carrying_enemy_flag()

False

In [9]:
def find_enemy_flags():
    flags = []
    if enemy_banner is None:
        return flags
    for x, y, z in find_blocks_by_name(enemy_banner):
        if my_team == "L" and x > 0:
            flags.append({"goal": (x, y - 1, z)})
        if my_team == "R" and x < 0:
            flags.append({"goal": (x, y - 1, z)})
    return flags

find_enemy_flags()

[{'goal': (-22, 1, -30)},
 {'goal': (-22, 1, -26)},
 {'goal': (-22, 1, -22)},
 {'goal': (-22, 1, -18)},
 {'goal': (-22, 1, -14)},
 {'goal': (-22, 1, -10)},
 {'goal': (-22, 1, -6)}]

In [10]:
def find_home_goals():
    homes = []
    if my_team is None:
        return homes
    enemy_flag_positions = set(find_blocks_by_name(enemy_banner))
    for x, y, z in find_blocks_by_name("gold_block"):
        if my_team == "L" and x < 0 and (x, y + 2, z) not in enemy_flag_positions:
            homes.append({"goal": (x, y + 1, z)})
        if my_team == "R" and x > 0 and (x, y + 2, z) not in enemy_flag_positions:
            homes.append({"goal": (x, y + 1, z)})
    return homes

find_home_goals()

[{'goal': (22, 1, 6)},
 {'goal': (22, 1, 10)},
 {'goal': (22, 1, 14)},
 {'goal': (22, 1, 18)},
 {'goal': (22, 1, 22)},
 {'goal': (22, 1, 26)},
 {'goal': (22, 1, 30)},
 {'goal': (22, 1, 34)}]

## 7. 走一步


In [11]:
def dist(x1, y1, z1, x2, y2, z2):
    dx = x1 - x2
    dy = y1 - y2
    dz = z1 - z2
    return dx * dx + dy * dy + dz * dz
    
def dist_from_bot_to(goal):
    return dist(
        float(bot.entity.position.x),
        float(bot.entity.position.y),
        float(bot.entity.position.z),
        (goal[0]),
        (goal[1]),
        (goal[2]))

def explore_goal():
    x = int(round(float(bot.entity.position.x))) + random.randint(-16, 16)
    y = int(round(float(bot.entity.position.y)))
    z = int(round(float(bot.entity.position.z))) + random.randint(-16, 16)
    return (x, y, z)

def step(last_step=None):
    view = {
        "team": my_team,
        "position": {"x": bot.entity.position.x, "y": bot.entity.position.y, "z": bot.entity.position.z},
        "carrying": carrying_enemy_flag(),
        "enemy_flags": find_enemy_flags(),
        "home_goals": find_home_goals(),
    }

    if view["carrying"] and view["home_goals"]:
        target = min(view["home_goals"], key=lambda item: dist_from_bot_to(item["goal"]))
        goal = pathfinder.goals.GoalBlock(*target["goal"])
        mode = "go_home"
    elif view["enemy_flags"]:
        target = min(view["enemy_flags"], key=lambda item: dist_from_bot_to(item["goal"]))
        goal = pathfinder.goals.GoalBlock(*target["goal"])
        mode = "get_flag"
    else:
        target = {"goal": explore_goal()}
        goal = pathfinder.goals.GoalBlock(*target["goal"])
        mode = "explore"
    # only setGoal if @goal changed
    if last_step is None or last_step["target"]["goal"] != target["goal"]:
        bot.pathfinder.setGoal(goal)
    return {"mode": mode, "target": target, "view": view}

step()


{'mode': 'get_flag',
 'target': {'goal': (-22, 1, -6)},
 'view': {'team': 'R',
  'position': {'x': 22.5, 'y': 1, 'z': 0.5},
  'carrying': False,
  'enemy_flags': [{'goal': (-22, 1, -30)},
   {'goal': (-22, 1, -26)},
   {'goal': (-22, 1, -22)},
   {'goal': (-22, 1, -18)},
   {'goal': (-22, 1, -14)},
   {'goal': (-22, 1, -10)},
   {'goal': (-22, 1, -6)}],
  'home_goals': [{'goal': (22, 1, 6)},
   {'goal': (22, 1, 10)},
   {'goal': (22, 1, 14)},
   {'goal': (22, 1, 18)},
   {'goal': (22, 1, 22)},
   {'goal': (22, 1, 26)},
   {'goal': (22, 1, 30)},
   {'goal': (22, 1, 34)}]}}

In [12]:
step()

{'mode': 'get_flag',
 'target': {'goal': (-22, 1, -6)},
 'view': {'team': 'R',
  'position': {'x': 22.5, 'y': 1, 'z': 0.5},
  'carrying': False,
  'enemy_flags': [{'goal': (-22, 1, -30)},
   {'goal': (-22, 1, -26)},
   {'goal': (-22, 1, -22)},
   {'goal': (-22, 1, -18)},
   {'goal': (-22, 1, -14)},
   {'goal': (-22, 1, -10)},
   {'goal': (-22, 1, -6)}],
  'home_goals': [{'goal': (22, 1, 6)},
   {'goal': (22, 1, 10)},
   {'goal': (22, 1, 14)},
   {'goal': (22, 1, 18)},
   {'goal': (22, 1, 22)},
   {'goal': (22, 1, 26)},
   {'goal': (22, 1, 30)},
   {'goal': (22, 1, 34)}]}}

## 8. 连续行动 (demo时不要运行)

In [13]:
import time
def continuous_move(wait_in_seconds = 100):
    wait_count = 0
    # 等待游戏开始，每 1s 检查一次
    while (not game_start) and (wait_count < wait_in_seconds):
        wait_count = wait_count + 1
        time.sleep(1)
    if not game_start:
        print("Timeout to start game")
        return
    print("Game started!")

    game_step = 0
    last_step = None
    while game_start:
        last_step = step(last_step)
        print(game_step, ":", last_step["target"], last_step["view"]["position"])
        game_step = game_step + 1
        time.sleep(1)
    print("Game over!")
        

In [14]:
continuous_move()

Game started!
0 : {'goal': (-22, 1, -6)} {'x': 12.874887372654308, 'y': 1, 'z': -0.49975590230965755}
1 : {'goal': (-22, 1, -6)} {'x': 5.896441803268744, 'y': 1, 'z': -2.494913340686264}
2 : {'goal': (-22, 1, -6)} {'x': -1.7206389110226012, 'y': 1, 'z': -3.499404593928497}
3 : {'goal': (-22, 1, -6)} {'x': -9.577909476930966, 'y': 1, 'z': -3.499738126098347}
4 : {'goal': (-22, 1, -6)} {'x': -17.293451615303226, 'y': 1, 'z': -4.504762972864557}
red_banner
5 : {'goal': (22, 1, 6)} {'x': -21.357303262828705, 'y': 1, 'z': -5.552888978842005}
red_banner
6 : {'goal': (22, 1, 6)} {'x': -17.566477101824017, 'y': 1, 'z': -3.680692746599711}
red_banner
7 : {'goal': (22, 1, 6)} {'x': -10.427707920968684, 'y': 1, 'z': -1.500475212031775}
red_banner
8 : {'goal': (22, 1, 6)} {'x': -2.289821817498817, 'y': 1, 'z': -1.5001416798626586}
red_banner
9 : {'goal': (22, 1, 6)} {'x': 5.667594547841342, 'y': 1, 'z': -0.413623343879799}
red_banner
10 : {'goal': (22, 1, 6)} {'x': 10.934795561096422, 'y': 1, 'z':

## 9. 停止 Bot


In [13]:
bot.quit("Notebook stop")